In [ ]:
import pandas as pd

In [19]:
# endereço atual
base = r"C:\Users\vini7\Desktop\Arquivos Python\Estudo de Caso Lighthouse 2026\Estudo-de-Caso-Lighthouse-2026\data"

# Carrega cada csv
orders = pd.read_csv(base + r"\orders.csv")
order_items = pd.read_csv(base + r"\order_items.csv")
products = pd.read_csv(base + r"\products.csv")
product_variants = pd.read_csv(base + r"\product_variants.csv")

In [21]:
# Dataset unificado com todos os dados necesários
dataset = (
    # order_items -> product_variants
    order_items
    .merge(
        product_variants,
        left_on = "product_variant_id",
        right_on = "id",
        how = "inner",
        # suffixes = ("_item", "_variant")
    )# .drop(columns = "id")
    # product_variants -> product
    .merge(
        products,
        left_on = "product_id",
        right_on = "id",
        how = "inner",
        # suffixes = ("_variant", "_product") 
    ).drop(columns = "id")
    # product -> order
    .merge(
        orders,
        left_on = "order_id",
        right_on = "id",
        how = "inner",
        # suffixes = ("_item", "_order") 
    ).drop(columns = "id")
)

# Convertendo placed_at para datetime
dataset["placed_at"] = pd.to_datetime(dataset["placed_at"])

# Selecionando vendas pagas e confirmadas da Bússola de Bordo 702
dataset = dataset[
    (dataset["name"] == "Bússola de Bordo 702") &
    (dataset["status"] .isin(["paid", "confirmed"])
     ).copy()
]

dataset

,id_x,order_id,product_variant_id,quantity,unit_price,icms_rate_x,ipi_rate_x,line_total,id_y,product_id,...,customer_id,salesperson_id,location_id,status,subtotal,discount_amount,total,placed_at,created_at,updated_at
452,457,150,148,2,1235.13,17.0,10.0,2470.26,148,74,...,1971,6.0,4,paid,32421.38,0.00,32421.38,2025-01-13 05:12:49,2025-01-13 05:12:49,2025-01-13 05:12:49
1010,1020,328,147,1,3400.77,17.0,5.0,3400.77,147,74,...,1012,9.0,6,paid,13755.20,1513.07,12242.13,2023-03-07 03:00:18,2023-03-07 03:00:18,2023-03-07 03:00:18
1208,1222,397,148,7,1235.13,17.0,10.0,8645.91,148,74,...,469,6.0,5,paid,46885.00,0.00,46885.00,2020-02-13 02:02:06,2020-02-13 02:02:06,2020-02-13 02:02:06
1634,1652,533,147,3,3400.77,17.0,5.0,10202.31,147,74,...,1834,NaN,5,confirmed,11658.87,0.00,11658.87,2024-06-20 11:12:42,2024-06-20 11:12:42,2024-06-20 11:12:42
1716,1737,563,148,1,1235.13,17.0,10.0,1235.13,148,74,...,562,10.0,4,paid,2049.18,0.00,2049.18,2024-03-22 15:47:59,2024-03-22 15:47:59,2024-03-22 15:47:59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145346,148295,49308,486,5,1758.87,18.0,0.0,8794.35,486,240,...,1738,NaN,2,paid,15716.91,0.00,15716.91,2020-12-17 18:07:48,2020-12-17 18:07:48,2020-12-17 18:07:48
145423,148375,49338,486,10,1758.87,18.0,0.0,17588.70,486,240,...,318,10.0,5,confirmed,55372.34,1107.45,54264.89,2025-06-07 06:27:52,2025-06-07 06:27:52,2025-06-07 06:27:52
145720,148675,49437,147,3,3400.77,17.0,5.0,10202.31,147,74,...,1642,6.0,5,paid,41488.95,0.00,41488.95,2026-02-09 15:20:14,2026-02-09 15:20:14,2026-02-09 15:20:14
146393,149359,49668,486,3,1758.87,18.0,0.0,5276.61,486,240,...,1815,9.0,3,paid,23788.40,0.00,23788.40,2024-12-30 02:34:28,2024-12-30 02:34:28,2024-12-30 02:34:28


In [ ]:
# Criando a coluna mês
dataset["mes"] = dataset["placed_at"].dt.to_period("M")

# Agrupamento de vendas por mes
vendas_mensais = (
    dataset.groupby("mes", as_index = False)["quantity"]
    .sum()
    .rename(columns={"quantity": "vendas"})
)

# Criação do calendário mensal
data_inicio = vendas_mensais["mes"].min()

calendario = pd.period_range(
    start = data_inicio,
    end = pd.Period("2026-03", freq = "M"),
    freq = "M"
)

serie = pd.DataFrame({"mes": calendario})

# Realiza LEFT JOIN das datas com a vendas, substituindo valores nulos por 0 (datas sem venda)
serie = serie.merge(vendas_mensais, on = "mes", how = "left")
serie["vendas"] = serie["vendas"].fillna(0)

serie

,mes,vendas
0,2020-01,29.0
1,2020-02,16.0
2,2020-03,17.0
3,2020-04,28.0
4,2020-05,5.0
...,...,...
70,2025-11,54.0
71,2025-12,19.0
72,2026-01,76.0
73,2026-02,55.0


In [23]:
# Dados para treino 
treino = serie[serie["mes"] <= pd.Period("2025-12", freq = "M")].copy()

# Dados para teste
teste = serie[
    (serie["mes"] >= pd.Period("2026-01", freq = "M")) &
    (serie["mes"] <= pd.Period("2026-03", freq = "M"))
].copy()

In [24]:
# Média móvel dos três meses anteriores
serie["previsao"] = (
    serie["vendas"].shift(1).rolling(window = 3).mean()
)

In [27]:
# Resultado da previsão
resultado = serie[
    (serie["mes"] >= pd.Period("2026-01", freq = "M")) &
    (serie["mes"] <= pd.Period("2026-03", freq = "M"))
]

# Cálculo do erro absoluto
resultado["erro_absoluto"] = (
    resultado["vendas"] - resultado["previsao"]
).abs()

# MAE
mae = resultado["erro_absoluto"].mean()

mae

np.float64(16.555555555555557)

In [28]:
# Soma das previsões do primeiro trimestre
soma_previsoes = resultado["previsao"].sum()
soma_previsoes_inteira = round(soma_previsoes)

soma_previsoes

np.float64(132.33333333333331)

In [30]:
print("Previsão de Demanda do Produto Bússola de Bordo 702")
print("\nResultados: ")
print(resultado[["mes", "vendas", "previsao", "erro_absoluto"]].to_string(index = False))
print(f"\nMAE: {mae:.2f} unidades")
print(f"soma das previsoes: {soma_previsoes_inteira} unidades")

Previsão de Demanda do Produto Bússola de Bordo 702

Resultados: 
    mes  vendas  previsao  erro_absoluto
2026-01    76.0 32.666667      43.333333
2026-02    55.0 49.666667       5.333333
2026-03    51.0 50.000000       1.000000

MAE: 16.56 unidades
soma das previsoes: 132 unidades
